# Intermediate 01 — Probability Foundations at a Deeper Level

Practice notebook for the **"Probability Foundations at a Deeper Level"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Sigma-algebras from a partition

The PDF defines a **sigma-algebra** on a sample space $\Omega$ as a collection $\mathcal{F}$ of subsets such that

1. $\Omega \in \mathcal{F}$,
2. if $A \in \mathcal{F}$ then $A^c \in \mathcal{F}$ (closure under complement),
3. if $A_1, A_2, \dots \in \mathcal{F}$ then $\bigcup_{n=1}^{\infty} A_n \in \mathcal{F}$ (closure under countable unions).

Intuitively $\mathcal{F}$ is the collection of all events we are *able to assign probabilities to*. For a **finite** $\Omega$, every sigma-algebra is generated by a **partition** $\mathcal{P} = \{B_1,\dots,B_k\}$ of $\Omega$: the sigma-algebra is the set of all unions of blocks of $\mathcal{P}$ (plus $\emptyset$).

Below we build a finite sigma-algebra from an explicit partition of $\Omega = \{1,2,3,4,5,6\}$ and verify the three axioms numerically.


In [ ]:
# Build a sigma-algebra from a partition of Omega = {1,...,6}
Omega = set(range(1, 7))                     # sample space (a die roll)
partition = [{1, 2}, {3, 4}, {5, 6}]          # blocks of the partition

def all_unions(partition):
    # Every sigma-algebra element is a union of some subset of blocks
    from itertools import combinations
    elems = [frozenset()]
    for r in range(1, len(partition) + 1):
        for combo in combinations(partition, r):
            elems.append(frozenset().union(*combo))
    return elems

F = all_unions(partition)
print("Omega              =", sorted(Omega))
print("partition blocks   =", [sorted(b) for b in partition])
print()
print("Sigma-algebra F has", len(F), "events:")
for A in sorted(F, key=lambda s: (len(s), sorted(s))):
    print("  ", sorted(A))

# --- Axiom checks ---
# (1) Omega in F
assert Omega in F, "Omega not in F"

# (2) closure under complement
Omega_f = frozenset(Omega)
assert all(Omega_f - A in F for A in F), "F not closed under complement"

# (3) closure under finite (hence countable) unions
from itertools import combinations as comb2
for A, B in comb2(F, 2):
    assert A | B in F, f"F not closed under union: {A} | {B}"
print()
print("Axioms 1-3 verified: Omega in F, closed under complement, closed under finite unions.")
print("|F| = 2^(#blocks) =", 2 ** len(partition), "(matches", len(F), "events)")


## Part 2 — Probability measure consequences, checked numerically

A **probability measure** is a function $P : \mathcal{F} \to [0,1]$ with

1. $P(\Omega) = 1$,
2. countable additivity: for disjoint $A_1, A_2, \dots \in \mathcal{F}$, $P\big(\bigcup_n A_n\big) = \sum_n P(A_n)$.

The PDF notes this measure-theoretic view unifies discrete and continuous probability: point masses and density integrals are both special cases of assigning consistent "sizes" to sets.

From these two axioms everything else follows — **monotonicity** ($A\subseteq B \Rightarrow P(A)\leq P(B)$), **complement rule** ($P(A^c)=1-P(A)$), **inclusion-exclusion** ($P(A\cup B)=P(A)+P(B)-P(A\cap B)$), and **Boole's inequality**. We verify these on the partition-based $\mathcal{F}$ from Part 1 by assigning a mass to each block.


In [ ]:
# Assign a probability mass to each partition block; induce P on all of F.
# Block probabilities must sum to 1 (this is P(Omega) = 1).
block_probs = {frozenset({1, 2}): 0.50,
               frozenset({3, 4}): 0.30,
               frozenset({5, 6}): 0.20}

def P(A):
    # P(A) = sum of masses of blocks contained in A
    return sum(m for block, m in block_probs.items() if block <= A)

# (1) P(Omega) = 1
Omega_f = frozenset(range(1, 7))
print("P(Omega)                =", round(P(Omega_f), 6))

# (2) Countable additivity on the disjoint blocks
total = sum(P(b) for b in block_probs)
print("sum P(block)            =", round(total, 6), "(should be 1)")

# Consequences
A = frozenset({1, 2, 3, 4})   # union of two blocks
B = frozenset({3, 4, 5, 6})   # union of two other blocks
print()
print("P(A)                    =", round(P(A), 4))
print("P(B)                    =", round(P(B), 4))
print("P(A^c)                  =", round(P(Omega_f - A), 4), "= 1 - P(A) =", round(1 - P(A), 4))
print("P(A u B) inclusion-excl  =", round(P(A | B), 4),
      "vs P(A)+P(B)-P(A&B)     =", round(P(A) + P(B) - P(A & B), 4))
print("P(A & B)                =", round(P(A & B), 4))

# Monotonicity: A subset B => P(A) <= P(B)
C = frozenset({1, 2})         # subset of A
print("P(C) <= P(A)?           ", P(C) <= P(A), f"({P(C):.2f} <= {P(A):.2f})")

# Boole's inequality (union bound): P(A u B) <= P(A) + P(B)
print("Boole: P(A u B) <= P(A)+P(B)?", P(A | B) <= P(A) + P(B),
      f"({P(A | B):.2f} <= {P(A) + P(B):.2f})")


## Part 3 — Distribution functions and the mixture example

For a real-valued random variable $X$, the **cumulative distribution function** (CDF) is

$$
F_X(x) = P(X \leq x), \quad x \in \mathbb{R}.
$$

Key properties from the PDF:

- $F_X$ is **non-decreasing**,
- $\lim_{x\to -\infty} F_X(x) = 0$ and $\lim_{x\to +\infty} F_X(x) = 1$,
- $F_X$ is **right-continuous**.

For discrete $X$, $F_X$ is a step function; for continuous $X$ with pdf $f_X$, $F_X(x) = \int_{-\infty}^{x} f_X(t)\,dt$.

The PDF's mixture example makes the point that the CDF is the *most general* description of a distribution: a strategy's daily return $R$ equals $0$ with probability $0.9$ and $Z\sim \mathcal{N}(0.02, 0.01^2)$ with probability $0.1$. Then $R$ is neither purely discrete nor purely continuous, with

$$
F_R(x) = 0.9\, \mathbf{1}_{\{x \geq 0\}} + 0.1\, F_Z(x).
$$

We plot $F_R$ and confirm the three CDF properties numerically.


In [ ]:
# Mixture CDF: 0.9 * indicator(x>=0) + 0.1 * Phi((x-0.02)/0.01)
p_atom, mu_Z, sd_Z = 0.9, 0.02, 0.01

def F_R(x):
    x = np.asarray(x, dtype=float)
    atom = np.where(x >= 0, 1.0, 0.0)
    cont = stats.norm.cdf(x, loc=mu_Z, scale=sd_Z)
    return p_atom * atom + (1 - p_atom) * cont

xs = np.linspace(-0.05, 0.10, 2001)
Fx = F_R(xs)

# --- Verify the three CDF properties ---
print("F_R(-inf) ~", round(F_R(-np.inf), 6), "(should be 0)")
print("F_R(+inf) ~", round(F_R(+np.inf), 6), "(should be 1)")
print("F non-decreasing?  ", bool(np.all(np.diff(Fx) >= -1e-12)),
      "(diffs >= 0 up to numerical noise)")

# Right-continuity at the atom x=0: F(0) should equal lim_{x -> 0+} F(x)
eps = 1e-9
print("Right-cont. at 0: F(0) =", round(F_R(0.0), 6),
      " vs F(0+eps) =", round(F_R(eps), 6), "(equal up to noise:",
      abs(float(F_R(0.0)) - float(F_R(eps))) < 1e-6, ")")
# The left limit at 0 is strictly smaller (the jump is the atom mass 0.9)
print("Size of jump at 0    =", round(float(F_R(0.0)) - float(F_R(-eps)), 6),
      "(should be ~0.9 = the atom mass)")

# Plot
fig, ax = plt.subplots()
ax.plot(xs, Fx, lw=2, label=r"$F_R(x) = 0.9\,\mathbf{1}_{x\geq 0} + 0.1\,F_Z(x)$")
ax.axvline(0, color="k", linestyle=":", alpha=0.5)
ax.axhline(0.9, color="r", linestyle="--", alpha=0.4, label="atom mass 0.9 at $x=0$")
ax.set_xlabel("$x$"); ax.set_ylabel("$F_R(x)$")
ax.set_title("Mixture CDF: 90% point mass at 0 + 10% Normal(0.02, 0.01^2)")
ax.legend(); plt.tight_layout(); plt.show()


## Part 4 — Inverse-CDF transform: uniform to any distribution

A cornerstone of simulation: if $U \sim \mathrm{Uniform}(0,1)$ and $F$ is a valid CDF with generalized inverse $F^{-1}(u) = \inf\{x : F(x) \geq u\}$, then

$$
X = F^{-1}(U) \quad \text{has CDF } F.
$$

So a single uniform source can be **transformed** into *any* distribution we want. We demonstrate this with the **exponential**$(\lambda)$ distribution, whose CDF is $F(x) = 1 - e^{-\lambda x}$ for $x \geq 0$ and whose inverse-CDF is explicit:

$$
F^{-1}(u) = -\frac{1}{\lambda}\log(1 - u).
$$

We draw uniforms, push them through $F^{-1}$, and compare the resulting empirical CDF and histogram against the theoretical exponential.


In [ ]:
lam = 1.7
rng = np.random.default_rng(101)
N = 200_000

# Step 1: uniform source
U = rng.uniform(0, 1, size=N)

# Step 2: inverse-CDF of the exponential(λ)
F_inv = lambda u: -np.log1p(-u) / lam
X = F_inv(U)

# Compare to theory
xs = np.linspace(0, X.max(), 400)
F_theory = 1 - np.exp(-lam * xs)
F_emp = np.array([np.mean(X <= x) for x in xs[::20]])

print("E[X] sample  =", round(X.mean(), 5), " vs theory 1/λ =", round(1 / lam, 5))
print("Var(X) sample =", round(X.var(ddof=1), 5), " vs theory 1/λ^2 =", round(1 / lam**2, 5))
print("max |F_emp - F_theory| =", round(np.max(np.abs(F_emp - F_theory[::20])), 5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# Histogram vs pdf
axes[0].hist(X, bins=80, density=True, alpha=0.5, label="inverse-CDF samples")
axes[0].plot(xs, lam * np.exp(-lam * xs), "r-", lw=2, label=r"$f_X(x)=\lambda e^{-\lambda x}$")
axes[0].set_xlabel("$x$"); axes[0].set_ylabel("density")
axes[0].set_title(f"Inverse-CDF samples vs Exponential(λ={lam}) pdf")
axes[0].legend()
# Empirical vs theoretical CDF
axes[1].plot(xs[::20], F_emp, "b-", lw=2, label="empirical CDF")
axes[1].plot(xs, F_theory, "r--", lw=2, label=r"$F(x)=1-e^{-\lambda x}$")
axes[1].set_xlabel("$x$"); axes[1].set_ylabel("$F_X(x)$")
axes[1].set_title("CDFs agree")
axes[1].legend()
plt.tight_layout(); plt.show()


## Part 5 — Change-of-variables: log-transform of an exponential

The PDF's transformation formula: if $Y = g(X)$ with $g$ differentiable and monotone, then

$$
f_Y(y) = f_X\big(g^{-1}(y)\big)\,\left|\frac{d}{dy}g^{-1}(y)\right|.
$$

For the **log-transform** of an Exponential$(\lambda)$ variable, $Y = \log X$ with $X = e^Y$, so $g^{-1}(y) = e^y$ and $\frac{d}{dy}g^{-1}(y) = e^y$. Hence

$$
f_Y(y) = \lambda e^{-\lambda e^{y}}\, e^{y}, \quad y \in \mathbb{R},
$$

a distribution with a heavy right tail in $y$-space (useful in log-return modeling).

We draw the exponential sample from Part 4, apply $Y = \log X$, and overlay the theoretical density $f_Y(y)$ derived above.

**Your turn:** try $Y = \sqrt{X}$ instead. Derive $f_Y(y)$ from the change-of-variables formula and check your derivation against a histogram of the transformed sample.


In [ ]:
# Reuse the exponential sample from Part 4 (seed 101, lam=1.7)
lam = 1.7
rng = np.random.default_rng(101)
U = rng.uniform(0, 1, size=200_000)
X = -np.log1p(-U) / lam           # Exponential(λ)
Y = np.log(X)                     # Y = log X

ys = np.linspace(Y.min(), Y.max(), 400)
# Theoretical density of Y = log X: f_Y(y) = λ * exp(-λ * e^y) * e^y
f_Y_theory = lam * np.exp(-lam * np.exp(ys)) * np.exp(ys)

print("E[Y] sample  =", round(Y.mean(), 5), " vs theory E[log X] =", round(-np.log(lam) - np.euler_gamma, 5))
print("(Euler-Mascheroni constant γ ≈ 0.5772; exponential log-mean = -log λ - γ.)")

fig, ax = plt.subplots()
ax.hist(Y, bins=80, density=True, alpha=0.5, label=r"sample of $Y=\log X$")
ax.plot(ys, f_Y_theory, "r-", lw=2, label=r"$f_Y(y)=\lambda e^{-\lambda e^y} e^y$")
ax.set_xlabel("$y$"); ax.set_ylabel("density")
ax.set_title(rf"Density of $Y=\log X$ with $X\sim$Exponential(λ={lam})")
ax.legend(); plt.tight_layout(); plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let $\Omega = \{a, b, c, d\}$ and partition $\mathcal{P} = \{\{a,b\}, \{c\}, \{d\}\}$. List every event in the sigma-algebra $\mathcal{F} = \sigma(\mathcal{P})$ and confirm that $|\mathcal{F}| = 2^{|\mathcal{P}|}$.

2. On the sigma-algebra from Problem 1, define $P$ by assigning masses $0.5, 0.3, 0.2$ to the three blocks. Verify numerically that $P(\Omega) = 1$, $P(A^c) = 1 - P(A)$ for every $A \in \mathcal{F}$, and that $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ for every pair $A, B$.

3. For the mixture CDF $F_R(x) = 0.9\,\mathbf{1}_{\{x \geq 0\}} + 0.1\,F_Z(x)$ with $Z \sim \mathcal{N}(0.02, 0.01^2)$, compute $P(R = 0)$, $P(R < 0)$, and $P(R > 0)$ analytically. Which property of $F_R$ reveals the point mass at $0$?

4. Using the inverse-CDF method with $U \sim \mathrm{Uniform}(0,1)$, give the explicit transform that produces a $\mathrm{Cauchy}(0, 1)$ sample (CDF $F(x) = \tfrac{1}{2} + \tfrac{1}{\pi}\arctan x$). Implement it in NumPy and confirm the sample median is near $0$ and the IQR matches the theoretical value $2$.

5. If $X \sim \mathrm{Exponential}(\lambda)$ and $Y = \sqrt{X}$, derive $f_Y(y)$ using the change-of-variables formula. State the support of $Y$ and check that $\int f_Y(y)\,dy = 1$.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The sigma-algebra consists of every union of a subset of the blocks. With blocks $B_1=\{a,b\}, B_2=\{c\}, B_3=\{d\}$ we get $2^3 = 8$ events:
$$\emptyset,\ \{a,b\},\ \{c\},\ \{d\},\ \{a,b,c\},\ \{a,b,d\},\ \{c,d\},\ \{a,b,c,d\} = \Omega.$$
So $|\mathcal{F}| = 8 = 2^{|\mathcal{P}|} = 2^3$, as required.

**2.** With block masses $0.5, 0.3, 0.2$ summing to $1$, we have $P(\Omega) = 1$.
For any $A \in \mathcal{F}$, $A^c = \Omega \setminus A$ is a union of the *remaining* blocks, so $P(A^c) = 1 - P(A)$. E.g. $A = \{a,b,c\}$ uses masses $0.5+0.3=0.8$, $A^c = \{d\}$ uses $0.2 = 1 - 0.8$.
For pairs, e.g. $A = \{a,b,c\}$ (mass $0.8$), $B = \{c,d\}$ (mass $0.3+0.2=0.5$), $A \cap B = \{c\}$ (mass $0.3$): $P(A \cup B) = P(\{a,b,c,d\}) = 1$, and $P(A)+P(B)-P(A\cap B) = 0.8 + 0.5 - 0.3 = 1.0$. The identity holds for every pair by finite additivity on disjoint blocks.

**3.** $P(R = 0) = F_R(0) - F_R(0^-) = 0.9$ (the jump at $0$, since $F_Z(0) = 0.5$ contributes $0.1 \cdot 0.5 = 0.05$ on both sides, which cancels in the difference). $P(R < 0) = F_R(0^-) = 0.1\,F_Z(0) = 0.05$. $P(R > 0) = 1 - F_R(0) = 1 - (0.9 + 0.1 \cdot 0.5) = 1 - 0.95 = 0.05$. The **jump discontinuity** of $F_R$ at $x = 0$ is exactly the point mass $P(R = 0) = 0.9$.

**4.** Inverting $F(x) = \tfrac{1}{2} + \tfrac{1}{\pi}\arctan x = u$ gives $x = \tan\big(\pi(u - \tfrac12)\big)$. So the transform is `X = np.tan(np.pi * (U - 0.5))`. The Cauchy median is $0$ and the IQR is $F^{-1}(0.75) - F^{-1}(0.25) = \tan(\pi/4) - \tan(-\pi/4) = 1 - (-1) = 2$. A NumPy simulation of e.g. $N = 200{,}000$ samples gives a sample median within $\approx 0.01$ of $0$ and a sample IQR within $\approx 0.02$ of $2$.

**5.** $g(y) = y^2$ on $y \geq 0$, so $g^{-1}(y) = \sqrt{y}$ (here using $y$ as the output variable; rename the output variable to $y \geq 0$) and $\left|\frac{d}{dy}g^{-1}(y)\right| = \frac{1}{2\sqrt{y}}$. With $f_X(x) = \lambda e^{-\lambda x}$ on $x \geq 0$,
$$f_Y(y) = \lambda e^{-\lambda \sqrt{y}}\,\frac{1}{2\sqrt{y}}, \quad y > 0.$$
Support is $y \in (0, \infty)$. The density integrates to $1$ because it is the change-of-variables image of a normalized density, or by direct substitution $x = \sqrt{y}$, $dx = \frac{dy}{2\sqrt{y}}$: $\int_0^\infty f_Y(y)\,dy = \int_0^\infty \lambda e^{-\lambda x}\,dx = 1$.

</details>
